In [ ]:
import pandas as pd

# inter-rater agreement metrics
from sklearn.metrics import cohen_kappa_score
from ira import calc_percentage_agreement, calc_bennetts_s_score

# Requirements Overlap

This notebook contains the calculation of the inter-rater agreement between the two independent raters determining quality defects on the *requirements* level (not on individual sentence level, which is done in [overlap-sentences.ipynb](./overlap-sentences.ipynb)).

## Data Preparation

First, load the data and prepare it for the overlap analysis.

In [ ]:
file_name: str = '../data/rq4tlr-manual-variables.xlsx'

# load the two ratings
rating1 = pd.read_excel(file_name, sheet_name='Requirements')
rating2 = pd.read_excel(file_name, sheet_name='Requirements Overlap')

# drop rows with missing values
rating1 = rating1.dropna(subset=['Rater'])
rating2 = rating2.dropna(subset=['Rater'])

The variable `factors` lists all columns of interest: these contain the ratings of the two raters in respect to each of the factors.

In [ ]:
factors: list[str] = ['Functional Duplication', 'Use Case Naming Problems', 
                      'Appropriate Scope', 'Cogent Text Order', 
                      'Consistent Level of Abstraction', 'Inputs and Outputs quantified', 
                      'Free of NFRs', 'Free of Actor-Actor Interaction',
                      'Free of justifications']

All of the ratings are boolean. Hence, the values are cast to type `bool`.

In [ ]:
for factor in factors:
    rating1[factor] = rating1[factor].astype(bool)
    rating2[factor] = rating2[factor].astype(bool)

Generate a unique ID per data entry which consists of the data set name (e.g., "eTours" or "iTrust") concatenated with the requirements name (e.g., "UC1") following the pattern `<dataset>-<requirement>` (e.g., "eTours-UC1").

In [ ]:
compose_id = lambda row: f'{row["Dataset"]}-{row["File"]}'

rating1['ID'] = rating1.apply(compose_id, axis=1)
rating2['ID'] = rating2.apply(compose_id, axis=1)

Filter the list of ratings from `rating1` (i.e., the complete, overall rating) to contain only those ratings that were also involved in the overlap.

In [ ]:
overlap: list[str] = rating2['ID'].values.tolist()

filtered_rating1 = rating1[rating1['ID'].isin(overlap)]

Finally, ensure that both data frames have the same order of ratings.

In [ ]:
rating2 = rating2.sort_values(by='ID').reset_index(drop=True)
filtered_rating1 = filtered_rating1.sort_values(by='ID').reset_index(drop=True)

## Calculating Agreement

We calculate the agreement of the ratings of both researchers by the means of percentage agreement[1], Cohen's Kappa [2], and Bennett's S-score [3]. Percentage agreement is the simplest type of inter-rater reliability. It suffers from the fact that it does not account for agreement caused by chance. Cohen's Kappa accounts for agreement caused by chance but samples the expected marginal distributions from the data directly. Bennett's S-score is a recommended alternative to Cohen's Kappa since it does account for agreement caused by chance but does assume an even marginal distribution. We report all three measures for completeness sake.

[1] Holsti, O. R. (1969). Content analysis for the social sciences and humanities. Reading. MA: Addison-Wesley (content analysis).

[2] Cohen, J. (1960). A coefficient of agreement for nominal scales. Educational and psychological measurement, 20(1), 37-46.

[3] Bennett, E. M., Alpert, R., & Goldstein, A. C. (1954). Communications through limited-response questioning. Public Opinion Quarterly, 18(3), 303-308.

In [ ]:
ira = dict()
ira['percentage_agreement'] = []
ira['cohens_kappa'] = []
ira['bennetts_s_score'] = []

print('Agreement per factor:')
for factor in factors: 
    percentage_agreement = calc_percentage_agreement(filtered_rating1[factor], rating2[factor])
    cohens_kappa = cohen_kappa_score(filtered_rating1[factor], rating2[factor], labels=[True, False])
    s_score = calc_bennetts_s_score(filtered_rating1[factor], rating2[factor], labels=[True, False])

    print(f'- {factor}: {percentage_agreement:.2%}, Cohen\'s Kappa of {cohens_kappa:.2%}, Bennett\'s S score of {s_score:.2%}')

    ira['percentage_agreement'].append(percentage_agreement)
    ira['cohens_kappa'].append(cohens_kappa)
    ira['bennetts_s_score'].append(s_score)

Finally, calculate the average agreement score for all three metrics across all `factors` of interest to obtain one final inter-rater agreement score (for each metric).

In [ ]:
averages = {key: sum(values) / len(values) for key, values in ira.items()}

for metric in averages:
    print(f'Average {metric}: {averages[metric]:.2%}')